If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec25_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 25: Center and Spread
v.ekc

Two numbers summarize a distribution: the **average** (center) and the **standard deviation** (spread). Today: what they mean, Chebyshev's guarantee, and standard units — the common ruler for all datasets. (Slides: KL Lecture 25 deck.)

**Today's sections:**
1. The average
2. The standard deviation
3. Chebyshev's bounds
4. Standard units
5. The SD and bell curves

In [ ]:
import matplotlib
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
import numpy as np
plots.style.use('fivethirtyeight')

---
## 1. The Average (Mean)

Four values, five ways to compute the same mean — including as a **weighted sum of proportions**, which is why histograms are all you need.

In [ ]:
values = make_array(2, 3, 3, 9)
values

In [ ]:
sum(values)/len(values)

In [ ]:
np.average(values)

In [ ]:
np.mean(values)

In [ ]:
(2 + 3 + 3 + 9)/4

In [ ]:
2*(1/4) + 3*(2/4) + 9*(1/4)

In [ ]:
2*0.25 + 3*0.5 + 9*0.25

In [ ]:
values_table = Table().with_columns('value', values)
values_table

In [ ]:
bins_for_display = np.arange(0.5, 10.6, 1)
values_table.hist('value', bins = bins_for_display)

Now 40 values with the *same proportions* — watch what happens:

In [ ]:
## Make array of 10 2s, 20 3s, and 10 9s

new_vals = make_array(2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
                      3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
                      9, 9, 9, 9, 9, 9, 9, 9, 9, 9)

In [ ]:
Table().with_column('value', new_vals).hist(bins = bins_for_display)

In [ ]:
np.average(values)

In [ ]:
np.average(new_vals)

In [ ]:
Table().with_column('value', new_vals).hist(bins = bins_for_display)
plots.ylim(-0.04, 0.5)
plots.plot([0, 10], [0, 0], color='grey', lw=2)
plots.scatter(4.25, -0.015, marker='^', color='red', s=100)
plots.title('Average as a Center of Gravity');

> **The mean depends only on the distribution** — same proportions, same mean, whether there are 4 values or 40. And on a histogram, the mean is the **balance point**; skewed tails drag it toward them (median resists — KL deck, slide 9).

---
## 2. The Standard Deviation

How far are values from the mean, *typically*? Build it in four steps:

### 📋 Board Reference

| Step | Code | Result |
|---|---|---|
| 1. deviations | `values - np.mean(values)` | they always sum to 0! |
| 2. square them | `deviations ** 2` | kills the canceling |
| 3. variance | `np.mean(deviations ** 2)` | mean squared deviation |
| 4. SD | `variance ** 0.5` or `np.std(values)` | back in the original units |

In [ ]:
sd_table = Table().with_columns('Value', values)
sd_table

In [ ]:
average_value = np.mean(values)
average_value

In [ ]:
deviations = values - average_value
sd_table = sd_table.with_column('Deviation', deviations)
sd_table

In [ ]:
sum(deviations)

In [ ]:
sd_table = sd_table.with_column('Squared Deviation', deviations ** 2)
sd_table

In [ ]:
# Variance of the data:
# mean squared deviation from average

variance = np.mean(deviations ** 2)
variance

In [ ]:
# Standard Deviation (SD): 
# root mean squared deviation from average
# = square root of the variance

sd = variance ** 0.5
sd

In [ ]:
np.std(values)

---
## 3. Chebyshev's Bounds

**No matter the shape** of the distribution, at least 1 − 1/z² of the data lies within z SDs of the mean (KL deck, slides 16–18): at least 75% within 2 SDs, at least 88.9% within 3.

Let's check on the maternal data — and note the observed proportions are usually much *higher* than the guarantee:

In [ ]:
births = Table.read_table('baby.csv').drop('Maternal Smoker')

In [ ]:
births.labels

In [ ]:
births.hist(overlay = False)

In [ ]:
mpw = births.column('Maternal Pregnancy Weight')
mean = np.mean(mpw)
sd = np.std(mpw)
mean, sd

In [ ]:
within_3_SDs = births.where(
    'Maternal Pregnancy Weight', are.between(mean - 3*sd, mean + 3*sd))

In [ ]:
# Proportion within 3 SDs of the mean

within_3_SDs.num_rows / births.num_rows

In [ ]:
# Chebyshev's bound: 
# This proportion should be at least

1 - 1/3**2

In [ ]:
births.labels

In [ ]:
# See if Chebyshev's bounds work for distributions with various shapes

for feature in births.labels:
    values = births.column(feature)
    mean = np.mean(values)
    sd = np.std(values)
    print()
    print(feature)
    for z in make_array(2, 3, 4, 5):
        chosen = births.where(feature, are.between(mean - z*sd, mean + z*sd))
        proportion = chosen.num_rows / births.num_rows
        percent = round(proportion * 100, 2)
        print('Average plus or minus', z, 'SDs:', percent, '%')

---
## 4. Standard Units

`(x − mean) / SD`: how many SDs above (+) or below (−) the mean. Every dataset converted to standard units has mean 0 and SD 1 — one ruler for everything.

In [ ]:
def standard_units(x):
    """Convert array x to standard units."""
    return (x - np.mean(x)) / np.std(x)

In [ ]:
ages = births.column('Maternal Age')
ages

In [ ]:
ages_standard_units = standard_units(ages)
ages_standard_units

In [ ]:
np.mean(ages_standard_units), np.std(ages_standard_units)


## Discussion Question 

In [ ]:
both = Table().with_columns(
    'Age in Years', ages,
    'Age in Standard Units', ages_standard_units
)
both

In [ ]:
np.mean(ages), np.std(ages)

In [ ]:
both.hist('Age in Years', bins = np.arange(15, 46, 2))

In [ ]:
both.hist('Age in Standard Units', bins = np.arange(-2.2, 3.4, 0.35))
plots.xlim(-2, 3.1);

> **Same histogram, new axis:** converting to standard units doesn't change the shape at all — only the labels. That's the point: 27 years old *is* +0.4 standard units.

---
## 5. The SD and Bell Curves 🔔

On a bell-shaped histogram you can *see* the SD: it's the distance from the mean to the **point of inflection** (where the curve switches from bulging out to bending in).

### Try It 1: Estimate by eye 👀

1. From the histogram below: the average is approximately…?

2. Locate the point of inflection on the right — the SD is approximately…? Then check both with code.

In [ ]:
births.hist('Maternal Height', bins = np.arange(56.5, 72.6, 1))
plots.xticks(np.arange(57, 72, 2));

In [ ]:
# 1–2. your eye-estimates, then check with np.mean / np.std


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Eyeball: mean ≈ 64 inches, inflection ≈ 66.5 → SD ≈ 2.5. The check:*

```python
heights = births.column('Maternal Height')
np.mean(heights), np.std(heights)     # (64.05, 2.52)
```

</details>

In [ ]:
heights = births.column('Maternal Height')
np.mean(heights), np.std(heights)

---
> **Today's takeaway:** mean = balance point, SD = typical distance from it, Chebyshev guarantees most data is within a few SDs *no matter what*, and standard units make every dataset comparable. Next: the one curve where the SD tells you everything.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — standard units
```python
def standard_units(x):
    return (x - np.mean(x)) / np.std(x)
```